<a href="https://colab.research.google.com/github/Abandonalo/SpatialGenUnity/blob/main/notebooks/Colab_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 · Mount Google Drive

ComfyUI, its models and its custom nodes live on Drive.


In [ ]:
# Mount Google Drive.

import os
import subprocess

MOUNT = "/content/drive"
DRIVE_ROOT = f"{MOUNT}/MyDrive"

os.chdir("/content")


def drive_usable():
    try:
        os.listdir(DRIVE_ROOT)
        return True
    except OSError:
        return False


if not drive_usable():
    print("♻️  Clearing a stale Drive mount…")
    subprocess.run(["fusermount", "-u", MOUNT], check=False,
                   capture_output=True)
    subprocess.run(["umount", "-l", MOUNT], check=False, capture_output=True)

from google.colab import drive
drive.mount(MOUNT, force_remount=True)

if not drive_usable():
    raise RuntimeError(
        f"{DRIVE_ROOT} is still not readable after remounting.\n"
        "Use Runtime > Disconnect and delete runtime, then run this cell first in the "
        "fresh runtime."
    )

print(f"✅ Drive mounted at {MOUNT}")
print(f"   cwd: {os.getcwd()}")


## Step 2 · ComfyUI and Python dependencies

Clones ComfyUI and pins the package versions the custom nodes need. **This is the cell to re-run if a dependency version is wrong.**


In [ ]:
%cd /content/drive/MyDrive

import os
import subprocess
import sys

# Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/drive/MyDrive/ComfyUI


!rm -f .git/index.lock

UPDATE_COMFY = False  # Set True only when you intentionally want to update ComfyUI.
if UPDATE_COMFY:
    subprocess.run(
        ["git", "stash", "push", "--include-untracked", "-m", "spatialgen-colab-autostash"],
        check=False,
    )
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    print("ℹ️ Skipping ComfyUI git pull for reproducible Colab startup")

# ----------------------------
# CUDA 12.6 check
# ----------------------------
def is_torch_cu126():
    try:
        import torch

        print("Found torch:", torch.__version__)
        print("CUDA build:", torch.version.cuda)

        return (
            torch.version.cuda == "12.6"
            and "+cu126" in torch.__version__
        )

    except Exception as e:
        print("Torch check failed:", e)
        return False


if is_torch_cu126():
    print("✅ CUDA 12.6 already installed — skipping")
else:
    print("⚠️ Installing CUDA 12.6 torch")

    !pip uninstall -y torch torchvision torchaudio

    !pip install \
        torch torchvision torchaudio \
        --index-url https://download.pytorch.org/whl/cu126

# Dependencies
!pip install -q -r requirements.txt
!pip install -q rembg onnxruntime
# NumPy ceiling is set by numba, which Hunyuan3D reaches through rembg -> pymatting
# and which refuses to import on NumPy >= 2.5. <2.6 let pip resolve 2.5 and the node
# failed to load.
!pip install -q --upgrade "numpy>=2.3,<2.5" "trimesh>=4.5.0" "transformers==4.41.2" "tokenizers==0.19.1" "diffusers==0.29.2" "peft==0.10.0" sentencepiece



import torch
print("✅ CUDA available:", torch.cuda.is_available())
print("💡 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("🔥 Torch CUDA:", torch.version.cuda)
print("📦 Torch:", torch.__version__)

## Step 3 · zrok tunnel setup

Installs the zrok binary and enables an environment for this runtime.


In [ ]:
import os
import re
import shutil
import subprocess
import tarfile
import urllib.request

# ============================================================
# Config
# ============================================================

ZROK_VERSION = "1.1.11"
ZROK_ENABLE_TOKEN = os.environ.get("ZROK_ENABLE_TOKEN", "").strip()
if not ZROK_ENABLE_TOKEN:
    try:
        from google.colab import userdata
        ZROK_ENABLE_TOKEN = (userdata.get("ZROK_ENABLE_TOKEN") or "").strip()
    except Exception:
        ZROK_ENABLE_TOKEN = ""
if not ZROK_ENABLE_TOKEN:
    import getpass
    ZROK_ENABLE_TOKEN = getpass.getpass("Enter zrok enable token: ").strip()
if not ZROK_ENABLE_TOKEN:
    raise RuntimeError("ZROK_ENABLE_TOKEN is required to create the Colab tunnel.")

archive_name = f"zrok_{ZROK_VERSION}_linux_amd64.tar.gz"
archive_url = (
    f"https://github.com/openziti/zrok/releases/download/"
    f"v{ZROK_VERSION}/{archive_name}"
)

extract_dir = "zrok_extract"

cache_dir = "/content/drive/MyDrive/.cache/zrok"
cache_binary = os.path.join(
    cache_dir,
    f"zrok_{ZROK_VERSION}"
)

local_binary = "/content/zrok"   # local disk: survives a Drive hiccup

os.makedirs(cache_dir, exist_ok=True)

# ============================================================
# Install / restore zrok binary
# ============================================================

if os.path.exists(cache_binary):
    print("✅ Reusing cached zrok binary from Drive")
    shutil.copy2(cache_binary, local_binary)

else:
    print("⬇️ Downloading zrok...")

    # Cleanup old artifacts
    for path in [archive_name, extract_dir]:
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

    urllib.request.urlretrieve(
        archive_url,
        archive_name
    )

    print("📦 Extracting zrok...")

    os.makedirs(extract_dir, exist_ok=True)

    with tarfile.open(
        archive_name,
        "r:gz"
    ) as tar:
        tar.extractall(extract_dir)

    zrok_binary = None

    for root, _, files in os.walk(extract_dir):
        if "zrok" in files:
            zrok_binary = os.path.join(root, "zrok")
            break

    if zrok_binary is None:
        raise FileNotFoundError(
            "Could not locate zrok binary after extraction."
        )

    shutil.copy2(zrok_binary, local_binary)
    shutil.copy2(zrok_binary, cache_binary)

# Make executable
os.chmod(local_binary, 0o755)

if os.path.exists(cache_binary):
    os.chmod(cache_binary, 0o755)

# Verify zrok exists
subprocess.run([local_binary, "version"])

# ============================================================
# zrok helpers
# ============================================================
# Shared with the tunnel cell further down. Every check asks the zrok CLI what state it
# is actually in rather than trusting an exit code: `zrok enable` can return 0 and still
# leave an environment that later commands refuse to load, which is what used to make
# this cell report success and the tunnel cell fail with "did you 'zrok enable'?".

import json
import queue
import threading

ZROK_BIN = os.path.abspath(local_binary)


def zrok(*args, timeout=180):
    """Runs the zrok CLI. Returns (returncode, combined stdout+stderr)."""
    res = subprocess.run([ZROK_BIN, *args], capture_output=True, text=True, timeout=timeout)
    return res.returncode, ((res.stdout or "") + "\n" + (res.stderr or "")).strip()


def zrok_json(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        return json.loads(text[start:])
    except json.JSONDecodeError:
        return None


def account_environments():
    """Environments registered on the account, or None when zrok cannot say."""
    rc, out = zrok("overview")
    data = zrok_json(out) if rc == 0 else None
    if data is None:
        return None
    return data.get("environments") or []


def environment_ready():
    """Whether this runtime has an environment the server will accept.

    Requires at least one registered environment, not merely that `zrok overview`
    answered. A deleted or expired environment leaves local state in ~/.zrok that
    still lets overview return an empty account, so "the command worked" is not
    evidence of authorisation — every later share request comes back 401.
    """
    return bool(account_environments())


def enable_environment(token, force=False):
    """Enables zrok and verifies it. Raises with the CLI output if it cannot."""
    home = os.path.expanduser("~/.zrok")

    if force and os.path.isdir(home):
        shutil.rmtree(home, ignore_errors=True)

    if environment_ready():
        return "already enabled"

    zrok("enable", token)
    if environment_ready():
        return "enabled"

    # A half-written environment cannot be repaired in place. Colab has no /dev/tty, so
    # any prompt zrok tries to show fails and leaves exactly that state.
    shutil.rmtree(home, ignore_errors=True)
    rc, out = zrok("enable", token)
    if environment_ready():
        return "enabled after reset"

    raise RuntimeError(
        "zrok enable did not produce a usable environment.\n"
        "Check the token is current at https://api.zrok.io and that the account has a "
        "free environment slot (each Colab runtime consumes one).\n\nzrok said:\n" + out
    )


print("🔑 zrok environment:", enable_environment(ZROK_ENABLE_TOKEN))


## Step 4 · Custom nodes and model checkpoints

Hunyuan3D 2.1, TripoSR, Essentials, plus the SD 1.5 and TripoSR weights.


In [ ]:
# ==========================================
# Hunyuan3D 2.1 + TripoSR + Essentials Custom Nodes
# ==========================================

import os
import re
import shutil
import sys
import subprocess
import urllib.request

COMFY_PATH = "/content/drive/MyDrive/ComfyUI"
CUSTOM_NODES = f"{COMFY_PATH}/custom_nodes"
HY3D_NODE = f"{CUSTOM_NODES}/ComfyUI-Hunyuan3d-2-1"
TRIPOSR_NODE = f"{CUSTOM_NODES}/ComfyUI-Flowty-TripoSR"
ESSENTIALS_NODE = f"{CUSTOM_NODES}/ComfyUI-essentials"
CHECKPOINT_DIR = f"{COMFY_PATH}/models/checkpoints"
TRIPOSR_MODEL_PATH = f"{CHECKPOINT_DIR}/TripoSRmodel.ckpt"
SD15_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/v1-5-pruned-emaonly.safetensors"
INPAINT_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/sd-v1-5-inpainting.ckpt"
CUSTOM_NODE_SETUP_STAMP = "/tmp/spatialgen_custom_nodes_setup_v2"
FORCE_CUSTOM_NODE_SETUP = False

print("🚀 Setting up ComfyUI custom nodes...")


def run(cmd):
    return subprocess.run(cmd, check=False)


def install_requirements_if_present(path, label):
    req_path = os.path.join(path, "requirements.txt")
    if os.path.exists(req_path):
        print(f"📦 Installing {label} dependencies...")
        run([sys.executable, "-m", "pip", "install", "-r", req_path])
    else:
        print(f"ℹ️ {label} has no requirements.txt")


def install_wheel_or_source(package_dir, label):
    dist_dir = os.path.join(package_dir, "dist")
    if os.path.exists(dist_dir):
        for file in os.listdir(dist_dir):
            if "linux" in file and file.endswith(".whl"):
                wheel_path = os.path.join(dist_dir, file)
                print(f"⚡ Installing {label} wheel: {file}")
                result = run([sys.executable, "-m", "pip", "install", wheel_path])
                if result.returncode == 0:
                    return
    print(f"🔧 Falling back to source build ({label})...")
    run([sys.executable, "-m", "pip", "install", package_dir])

def patch_triposr_numpy2_ptp(node_dir):
    """Patch old ndarray.ptp(...) calls for NumPy 2.x without downgrading NumPy globally."""
    if not os.path.isdir(node_dir):
        return

    ptp_pattern = re.compile(
        r"(?P<expr>\b[A-Za-z_][A-Za-z0-9_]*(?:\[[^\n\]]+\]|\.[A-Za-z_][A-Za-z0-9_]*)*)\.ptp\((?P<args>[^()\n]*)\)"
    )
    patched_files = []

    for root, _, files in os.walk(node_dir):
        for file_name in files:
            if not file_name.endswith(".py"):
                continue

            path = os.path.join(root, file_name)
            with open(path, "r", encoding="utf-8") as f:
                original = f.read()

            if ".ptp(" not in original:
                continue

            def replace_ptp(match):
                expr = match.group("expr")
                args = match.group("args").strip()
                return f"np.ptp({expr}{', ' + args if args else ''})"

            updated = ptp_pattern.sub(replace_ptp, original)
            if updated == original:
                continue

            if "import numpy as np" not in updated:
                updated = "import numpy as np\n" + updated

            with open(path, "w", encoding="utf-8") as f:
                f.write(updated)
            patched_files.append(os.path.relpath(path, node_dir))

    if patched_files:
        print("🩹 Patched Flowty TripoSR NumPy 2.x ptp usage:", ", ".join(patched_files))
    else:
        print("✅ Flowty TripoSR NumPy 2.x ptp patch not needed")


def patch_triposr_glb_export(node_dir):
    """Patch Flowty TripoSR viewer to honor a format input and export GLB."""
    init_path = os.path.join(node_dir, "__init__.py")
    if not os.path.exists(init_path):
        print("⚠️ Flowty TripoSR __init__.py not found; GLB export patch skipped")
        return

    with open(init_path, "r", encoding="utf-8") as f:
        original = f.read()

    if "SPATIALGEN_GLB_EXPORT_PATCH = True" in original:
        print("✅ Flowty TripoSR GLB export patch already applied")
        return

    viewer_class = '''class TripoSRViewer:
 SPATIALGEN_GLB_EXPORT_PATCH = True

 @classmethod
 def INPUT_TYPES(s):
  return {
   "required": {
    "mesh": ("MESH",),
    "format": (["obj", "glb"], {"default": "glb"})
   }
  }

 RETURN_TYPES = ()
 OUTPUT_NODE = True
 FUNCTION = "display"
 CATEGORY = "Flowty TripoSR"

 def display(self, mesh, format="glb"):
  saved = list()
  full_output_folder, filename, counter, subfolder, filename_prefix = get_save_image_path("meshsave",
  get_output_directory())
  filename_extension = str(format).lower()
  if filename_extension not in {"obj", "glb"}:
   filename_extension = "glb"

  transform = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, -1, 0, 0], [0, 0, 0, 1]])
  for (batch_number, single_mesh) in enumerate(mesh):
   filename_with_batch_num = filename.replace("%batch_num%", str(batch_number))
   file = f"{filename_with_batch_num}_{counter:05}_.{filename_extension}"
   mesh_to_export = single_mesh.copy()
   mesh_to_export.apply_transform(transform)
   mesh_to_export.export(path.join(full_output_folder, file), file_type=filename_extension)
   saved.append({
    "filename": file,
    "type": "output",
    "subfolder": subfolder
   })

  return {"ui": {"mesh": saved}}
'''

    pattern = r"class TripoSRViewer:\n.*?(?=\nNODE_CLASS_MAPPINGS\s*=)"
    updated, replacements = re.subn(pattern, viewer_class, original, count=1, flags=re.S)
    if replacements != 1:
        print("⚠️ Flowty TripoSR viewer class not found; GLB export patch skipped")
        return

    with open(init_path, "w", encoding="utf-8") as f:
        f.write(updated)
    print("🩹 Patched Flowty TripoSR viewer for GLB export")


def patch_hunyuan_trust_remote_code(node_dir):
    """Allow Hunyuan 2.1 PaintPBR custom UNet code to load with newer diffusers."""
    target_path = os.path.join(node_dir, "hy3dpaint", "utils", "multiview_utils.py")
    if not os.path.exists(target_path):
        print("⚠️ Hunyuan multiview_utils.py not found; trust_remote_code patch skipped")
        return False

    with open(target_path, "r", encoding="utf-8") as f:
        original = f.read()

    if "trust_remote_code=True" in original:
        print(f"✅ Hunyuan PaintPBR trust_remote_code patch already applied: {target_path}")
        return True

    pattern = r"(HunyuanPaintPipeline\.from_pretrained\(\s*model_path\s*,\s*torch_dtype\s*=\s*torch\.float16)(\s*,?\s*\))"
    updated, replacements = re.subn(
        pattern,
        r"\1,\n trust_remote_code=True\2",
        original,
        count=1,
        flags=re.S,
    )

    if replacements != 1:
        print("⚠️ Hunyuan PaintPBR loader shape changed; trust_remote_code patch skipped")
        print(f"   Checked: {target_path}")
        return False

    with open(target_path, "w", encoding="utf-8") as f:
        f.write(updated)

    with open(target_path, "r", encoding="utf-8") as f:
        verified = "trust_remote_code=True" in f.read()
    if verified:
        print(f"🩹 Patched Hunyuan PaintPBR loader with trust_remote_code=True: {target_path}")
    else:
        print(f"⚠️ Hunyuan PaintPBR patch write failed verification: {target_path}")
    return verified


if os.path.exists(CUSTOM_NODE_SETUP_STAMP) and not FORCE_CUSTOM_NODE_SETUP:
    print("✅ Reusing custom-node setup from this runtime")
else:
    # Install system build tools only if they are missing.
    if shutil.which("g++") and shutil.which("make"):
        print("✅ Build tools already available")
    else:
        print("🔧 Installing system build dependencies...")
        run(["apt-get", "update"])
        run(["apt-get", "install", "-y", "build-essential"])

    if not os.path.exists(HY3D_NODE):
        print("📦 Cloning Hunyuan3D 2.1...")
        run(["git", "clone", "https://github.com/visualbruno/ComfyUI-Hunyuan3d-2-1", HY3D_NODE])
    else:
        print("✅ Hunyuan3D already exists")
    patch_hunyuan_trust_remote_code(HY3D_NODE)

    if not os.path.exists(TRIPOSR_NODE):
        print("📦 Cloning Flowty TripoSR...")
        run(["git", "clone", "https://github.com/flowtyone/ComfyUI-Flowty-TripoSR", TRIPOSR_NODE])
    else:
        print("✅ Flowty TripoSR already exists")
    patch_triposr_glb_export(TRIPOSR_NODE)

    if not os.path.exists(ESSENTIALS_NODE):
        print("📦 Cloning ComfyUI Essentials...")
        run(["git", "clone", "https://github.com/cubiq/ComfyUI_essentials", ESSENTIALS_NODE])
    else:
        print("✅ Essentials already exists")

    install_requirements_if_present(HY3D_NODE, "Hunyuan")
    install_requirements_if_present(TRIPOSR_NODE, "TripoSR")
    install_requirements_if_present(ESSENTIALS_NODE, "Essentials")

    print("📦 Upgrading Trimesh for NumPy 2.x compatibility...")
    run([sys.executable, "-m", "pip", "install", "--upgrade", "trimesh>=4.5.0"])

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    if not os.path.exists(SD15_CHECKPOINT_PATH):
        print("⬇️ Downloading SD 1.5 checkpoint for TripoSR prompt image generation...")
        urllib.request.urlretrieve(
            "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly.safetensors",
            SD15_CHECKPOINT_PATH,
        )
    else:
        print("✅ SD 1.5 checkpoint already exists")

    if not os.path.exists(INPAINT_CHECKPOINT_PATH):
        print("⬇️ Downloading SD 1.5 inpainting checkpoint for refinement...")
        urllib.request.urlretrieve(
            "https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt",
            INPAINT_CHECKPOINT_PATH,
        )
    else:
        print("✅ SD 1.5 inpainting checkpoint already exists")

    if not os.path.exists(TRIPOSR_MODEL_PATH):
        print("⬇️ Downloading TripoSR checkpoint...")
        urllib.request.urlretrieve(
            "https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt",
            TRIPOSR_MODEL_PATH,
        )
    else:
        print("✅ TripoSR checkpoint already exists")

    print("🧠 Installing custom rasterizer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/custom_rasterizer", "rasterizer")

    print("🎨 Installing differentiable renderer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/DifferentiableRenderer", "renderer")

    with open(CUSTOM_NODE_SETUP_STAMP, "w", encoding="utf-8") as f:
        f.write("ok")

    print("✅ All custom nodes installed successfully!")

# The setup stamp can skip the large install branch on warm runtimes; still
# ensure the refinement-specific checkpoint exists before starting the server.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
if not os.path.exists(INPAINT_CHECKPOINT_PATH):
    print("⬇️ Downloading SD 1.5 inpainting checkpoint for refinement...")
    urllib.request.urlretrieve(
        "https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt",
        INPAINT_CHECKPOINT_PATH,
    )
else:
    print("✅ SD 1.5 inpainting checkpoint already exists")

patch_hunyuan_trust_remote_code(HY3D_NODE)
patch_triposr_glb_export(TRIPOSR_NODE)
print("📦 Ensuring Trimesh is NumPy 2.x compatible...")
run([sys.executable, "-m", "pip", "install", "--upgrade", "trimesh>=4.5.0"])
patch_triposr_numpy2_ptp(TRIPOSR_NODE)

🚀 Setting up ComfyUI custom nodes...
✅ Reusing custom-node setup from this runtime
🩹 Patched Hunyuan PaintPBR loader with trust_remote_code=True: /content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Hunyuan3d-2-1/hy3dpaint/utils/multiview_utils.py
✅ Flowty TripoSR GLB export patch already applied
📦 Ensuring Trimesh is NumPy 2.x compatible...
✅ Flowty TripoSR NumPy 2.x ptp patch not needed


## Step 5 · Stop any running ComfyUI


In [ ]:
!pkill -f main.py

## Step 6 · Launch ComfyUI

Serves on port 8188 and reports any custom node that failed to import.


In [ ]:
import os
import subprocess
import sys
import time
import torch

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        # numba (via rembg -> pymatting) rejects NumPy >= 2.5; see cell 2.
        "numpy>=2.3,<2.5",
        "trimesh>=4.5.0",
        # Keep TripoSR on the ViT/DINO key layout expected by the official checkpoint.
        "transformers==4.41.2",
        "tokenizers==0.19.1",
        "diffusers==0.29.2",
        "peft==0.10.0",
        "accelerate",
        "sentencepiece",
    ],
    check=True,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Ensure PyTorch defaults to CUDA
if torch.cuda.is_available():
    torch.set_default_device("cuda")
    print("✅ Using GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ CUDA not available — something is wrong")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

DISABLE_SMART_MEMORY = False
FORCE_FP32 = False


subprocess.run(["pkill", "-f", "main.py"], check=False)
time.sleep(2)
COMFY_LOG_PATH = "/tmp/spatialgen_comfyui.log"
COMFY_DB_PATH = "/tmp/spatialgen_comfyui.db"
open(COMFY_LOG_PATH, "w", encoding="utf-8").close()


launch_args = [
    "python",
    "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
    "--database-url", f"sqlite:///{COMFY_DB_PATH}",
]
if DISABLE_SMART_MEMORY:
    launch_args.append("--disable-smart-memory")
if FORCE_FP32:
    launch_args.append("--force-fp32")

comfy_log_handle = open(COMFY_LOG_PATH, "a", encoding="utf-8", buffering=1)
comfy_process = subprocess.Popen(
    launch_args,
    stdout=comfy_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = False
log_pos = 0
COMFY_START_TIMEOUT_SECONDS = 300
poll_interval = 0.5
for _ in range(int(COMFY_START_TIMEOUT_SECONDS / poll_interval)):
    with open(COMFY_LOG_PATH, "r", encoding="utf-8", errors="replace") as log_file:
        log_file.seek(log_pos)
        new_lines = log_file.readlines()
        log_pos = log_file.tell()

    for line in new_lines:
        print(line.strip())
        if "Starting server" in line or "To see the GUI go to" in line:
            ready = True

    if ready:
        break
    if comfy_process.poll() is not None:
        break

    time.sleep(poll_interval)

def report_failed_custom_nodes(log_path):
    """Names custom nodes that failed to import.

    ComfyUI starts happily without them and the problem only shows up later as an
    unknown node type in a workflow, so it is worth surfacing at launch.
    """
    try:
        with open(log_path, "r", encoding="utf-8", errors="replace") as handle:
            lines = handle.readlines()
    except OSError:
        return

    failed = [l.strip() for l in lines if "IMPORT FAILED" in l]
    if not failed:
        return

    print("\n⚠️ Some custom nodes did not load:")
    for line in failed:
        print(f"   {line}")
    reasons = [l.strip() for l in lines if "Cannot import" in l]
    for line in reasons[-3:]:
        print(f"   {line}")
    print("   Workflows using those nodes will fail with an unknown node type.")


if ready:
    print("✅ ComfyUI is running on port 8188")
    print(f"🧾 ComfyUI logs: {COMFY_LOG_PATH}")
    report_failed_custom_nodes(COMFY_LOG_PATH)

    # Ask a fresh interpreter rather than this kernel: after a pip downgrade the
    # kernel still holds the version it imported at startup, which is not what the
    # ComfyUI subprocess just loaded.
    probe = subprocess.run(
        [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
        capture_output=True, text=True)
    installed = probe.stdout.strip()
    if installed and tuple(int(p) for p in installed.split(".")[:2]) >= (2, 5):
        print(f"\n⚠️ NumPy {installed} is installed; numba needs 2.4 or less, so the")
        print("   Hunyuan3D nodes cannot import.")
        print("   Fix: run 'Step 2 · ComfyUI and Python dependencies' above, then")
        print("   Runtime > Restart session, then run the cells again from Step 2.")
else:
    exit_code = comfy_process.poll()
    if exit_code is None:
        print(f"⏳ ComfyUI is still starting after {COMFY_START_TIMEOUT_SECONDS}s; keep tailing {COMFY_LOG_PATH} or rerun the endpoint check cell shortly.")
    else:
        print(f"❌ ComfyUI exited before startup completed (exit code {exit_code}).")
    print(f"🧾 Last ComfyUI log lines from {COMFY_LOG_PATH}:")
    try:
        with open(COMFY_LOG_PATH, "r", encoding="utf-8", errors="replace") as log_file:
            for line in log_file.readlines()[-80:]:
                print(line.rstrip())
    except Exception as exc:
        print(f"Could not read ComfyUI log: {exc}")

## Step 7 · SpatialGen router

Clones this repo and imports `tools.comfy_router_backend`, so Colab and a local machine run the same router code.


In [ ]:
# ==========================================================
# SpatialGen router
# ==========================================================
# The router that Unity talks to lives in the repo, at
# tools/comfy_router_backend. This cell clones that repo and imports it, so
# Colab and a local machine run the exact same code. Editing the router means
# editing the repo, never this notebook.
# ==========================================================

import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request

REPO_URL = "https://github.com/Abandonalo/SpatialGenUnity.git"
REPO_PATH = "/content/SpatialGenUnity"
COMFY_PATH = "/content/drive/MyDrive/ComfyUI"
CONTROLNET_DIR = f"{COMFY_PATH}/models/controlnet"
WORKFLOW_DIR = f"{COMFY_PATH}/user/default/workflows"

# ---- 1. Get the router source ---------------------------------------------
# This repository is private, so Colab needs a token. Add one under the key icon in
# the left sidebar (Secrets) named GITHUB_TOKEN, with read access to the repo, and
# enable notebook access. A fine-grained token needs only "Contents: read".
def github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        return (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        return ""


def git_env():
    """Environment carrying the auth header, if a token is available.

    Passed as GIT_CONFIG_* rather than embedded in the remote URL or a -c flag: the
    URL form writes the token into .git/config, and -c puts it in the process
    command line where any `ps` can read it.
    """
    env = dict(os.environ)
    token = github_token()
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraheader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}",
        })
    env["GIT_TERMINAL_PROMPT"] = "0"      # fail fast instead of hanging on a prompt
    return env


def git_run(*args):
    """Runs git and raises with git's own message when it fails.

    subprocess check=True raises CalledProcessError with no detail, which turns a
    clone failure into a bare "exit status 128" and hides the one line that says why.
    """
    res = subprocess.run(["git", *args], capture_output=True, text=True, env=git_env())
    if res.returncode != 0:
        detail = ((res.stdout or "") + (res.stderr or "")).strip() or "(no output)"
        # Never echo the token, which would otherwise reach the notebook output.
        detail = detail.replace(github_token(), "***") if github_token() else detail
        if "could not read Username" in detail or "Authentication failed" in detail:
            detail += (
                "\n\nThis repository is private and Colab has no credentials for it."
                "\nAdd a GITHUB_TOKEN secret (key icon, left sidebar) with read access,"
                "\nenable notebook access for it, then re-run this cell."
            )
        raise RuntimeError(f"git {' '.join(args)} failed with {res.returncode}:\n{detail}")
    return res.stdout


def clear_path(path):
    """Removes `path`, and says why if it cannot."""
    shutil.rmtree(path, ignore_errors=True)
    if os.path.exists(path):
        # The quiet pass failed; repeat it loudly so the real errno surfaces rather
        # than leaving a non-empty directory for git clone to trip over.
        shutil.rmtree(path)


def sync_repo(url, path):
    if os.path.isdir(os.path.join(path, ".git")):
        try:
            git_run("-C", path, "fetch", "--depth", "1", "origin", "main")
            git_run("-C", path, "reset", "--hard", "origin/main")
            return "updated"
        except RuntimeError as exc:
            print(f"⚠️ Existing checkout could not be updated, re-cloning.\n{exc}\n")

    clear_path(path)
    git_run("clone", "--depth", "1", url, path)
    return "cloned"


if not github_token():
    print("ℹ️ No GITHUB_TOKEN secret found. If the clone fails with an auth error,")
    print("   add one via the key icon in the left sidebar.")

print(f"📥 SpatialGenUnity: {sync_repo(REPO_URL, REPO_PATH)}")
print(f"   at commit {git_run('-C', REPO_PATH, 'rev-parse', '--short', 'HEAD').strip()}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_PATH}/requirements.txt"],
    check=True,
)

# ---- 2. ControlNet weights -------------------------------------------------
# The proxy depth and edge maps Unity sends are useless without these; they are what
# turn a placed primitive into a spatial constraint on the diffusion.
#
# fp16 safetensors: 689 MB each rather than 1.4 GB for the .pth originals, which
# matters on a Colab runtime. Note the depth model is v11f1p, not v11p.
CONTROLNET_REPO = "https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main"
CONTROLNET_MODELS = {
    "controlnet-depth/control_v11f1p_sd15_depth_fp16.safetensors":
        f"{CONTROLNET_REPO}/control_v11f1p_sd15_depth_fp16.safetensors",
    "controlnet-canny/control_v11p_sd15_canny_fp16.safetensors":
        f"{CONTROLNET_REPO}/control_v11p_sd15_canny_fp16.safetensors",
}


def download(url, target):
    """Downloads to a .part file and renames only on success.

    urlretrieve leaves a truncated file behind when a transfer dies, and the next run
    then sees a non-empty file and skips it. An atomic rename makes "the file exists"
    mean "the download finished".
    """
    part = f"{target}.part"
    try:
        with urllib.request.urlopen(url, timeout=120) as response:
            expected = int(response.headers.get("Content-Length") or 0)
            with open(part, "wb") as out:
                shutil.copyfileobj(response, out)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"HTTP {exc.code} fetching {url}") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Could not reach {url}: {exc.reason}") from exc

    written = os.path.getsize(part)
    if expected and written != expected:
        os.remove(part)
        raise RuntimeError(f"{url} returned {written} bytes, expected {expected}")

    os.replace(part, target)
    return written


for file_name, url in CONTROLNET_MODELS.items():
    target = os.path.join(CONTROLNET_DIR, file_name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f"✅ {file_name} already present")
        continue
    print(f"⬇️  {file_name} …")
    size = download(url, target)
    print(f"✅ {file_name} ({size / 1048576:.0f} MB)")

# ---- 3. Workflow names -----------------------------------------------------
# The Hunyuan graph ships with its custom nodes rather than with this repo, so
# the router looks for it in ComfyUI's own workflow folder under the name it
# routes by. Alias whatever you exported from the ComfyUI editor to that name.
WORKFLOW_ALIASES = {
    "Full_Workflow_API.json": "generation_hunyuan.json",
    "Upload_Image_API.json": "generation_hunyuan_image.json",
}

os.makedirs(WORKFLOW_DIR, exist_ok=True)
for source_name, routed_name in WORKFLOW_ALIASES.items():
    source = os.path.join(WORKFLOW_DIR, source_name)
    routed = os.path.join(WORKFLOW_DIR, routed_name)
    if os.path.exists(source) and not os.path.exists(routed):
        shutil.copyfile(source, routed)
        print(f"🔗 {source_name} → {routed_name}")

# ---- 4. Router configuration ----------------------------------------------
os.environ.update({
    "COMFY_BASE_URL": "http://127.0.0.1:8188",
    "COMFY_ROOT": COMFY_PATH,
    "COMFY_INPUT_DIR": f"{COMFY_PATH}/input",
    "COMFY_CHECKPOINT": "v1-5-pruned-emaonly.safetensors",
    "COMFY_INPAINT_CHECKPOINT": "sd-v1-5-inpainting.ckpt",
    # Must match ControlNetLoader's dropdown, i.e. the path under models/controlnet.
    "COMFY_CONTROLNET_DEPTH": "controlnet-depth/control_v11f1p_sd15_depth_fp16.safetensors",
    "COMFY_CONTROLNET_CANNY": "controlnet-canny/control_v11p_sd15_canny_fp16.safetensors",
    "SPATIALGEN_TRIPO_MODEL": "TripoSRmodel.ckpt",
    "SPATIALGEN_GEOMETRY_RESOLUTION": "512",
    "SPATIALGEN_TRIPO_THRESHOLD": "25",
    # A GPU run is minutes, not seconds.
    "COMFY_TIMEOUT_SECONDS": "900",
})

# ---- 5. Import the app -----------------------------------------------------
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

for module in [name for name in sys.modules if name.startswith("tools.comfy_router_backend")]:
    del sys.modules[module]

from tools.comfy_router_backend.app import app  # noqa: E402

print("\n✅ Router ready. Endpoints:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"   {','.join(sorted(route.methods)):<6} {route.path}")


## Step 8 · Serve the router

On port 8000, which the tunnel points at.


In [ ]:
# Serve the router on port 8000 (the port the tunnel points at).
import socket
import threading
import time

import uvicorn

PORT = 8000


def port_is_free(port):
    """Whether we can bind `port` right now.

    Checked directly rather than by asking whether our previous thread is alive: a
    uvicorn whose thread has already exited can still be holding the socket, which is
    how re-running this cell ended in "address already in use".
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            probe.bind(("0.0.0.0", port))
            return True
        except OSError:
            return False


state = globals().setdefault("_spatialgen_server", {})

# Shut down a server this notebook started earlier. should_exit is uvicorn's own
# cooperative stop; there is no way to kill the thread otherwise.
previous = state.get("server")
if previous is not None:
    print("♻️  stopping the router started earlier in this session")
    previous.should_exit = True
    thread = state.get("thread")
    if thread is not None:
        thread.join(timeout=15)
    state.clear()
    for _ in range(15):
        if port_is_free(PORT):
            break
        time.sleep(1)

if not port_is_free(PORT):
    print(f"❌ Port {PORT} is held by a process this notebook cannot stop.")
    print("   Runtime > Restart session, then run the cells again from Step 7.")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info", loop="asyncio")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    state.update(server=server, thread=thread)

    for _ in range(60):
        if getattr(server, "started", False):
            break
        time.sleep(0.5)

    if getattr(server, "started", False):
        print(f"✅ Router running on port {PORT}")
    else:
        print("⚠️ Router did not report startup; see the log above.")


## Step 9 · Public tunnel for Unity

Produces the URL to paste into the Spatial Generation window.


In [ ]:
# ==========================================================
# Public tunnel for Unity
# ==========================================================
# Reuses the reserved share when it already points at the router, and repairs it when it
# does not. The two states that used to end the run here are handled explicitly:
#   * the local environment is gone or stale  -> re-enable and verify
#   * the name is reserved by an environment  -> release it and reserve again
#     this runtime cannot use (HTTP 409)
# ==========================================================

import time

TARGET = "http://127.0.0.1:8000"
RESERVED_NAME = "comfyuitunnel"

if "enable_environment" not in globals():
    raise RuntimeError("Run the zrok setup cell near the top of the notebook first.")


def looks_unauthorized(text):
    lowered = (text or "").lower()
    return "401" in lowered or "unauthorized" in lowered


def reserved_share_target(name):
    """Backend target of the reserved share, or None if this account cannot see it."""
    rc, out = zrok("overview")
    data = zrok_json(out) if rc == 0 else None
    if not data:
        return None
    # `or []` rather than a .get default: zrok emits "environments": null when the
    # account has none, and a default only applies to a missing key, not a null one.
    for env in data.get("environments") or []:
        for share in env.get("shares") or []:
            if share.get("token") == name:
                return (share.get("backendProxyEndpoint") or "").rstrip("/")
    return None


def account_summary():
    """Environment and share counts, for diagnosing quota errors."""
    envs = account_environments()
    if envs is None:
        return "could not read the account overview"
    shares = sum(len(e.get("shares") or []) for e in envs)
    return f"{len(envs)} environment(s), {shares} share(s) on this account"


def ensure_reserved(name, target):
    """Guarantees a reserved share called `name` pointing at `target`.

    Returns None when the reservation cannot be made, so the caller can fall back to an
    ephemeral share rather than failing the whole cell.
    """
    target = target.rstrip("/")
    current = reserved_share_target(name)

    if current == target:
        return "reused"

    if current is not None:
        # Reserved against a different port. Reusing it would tunnel Unity to the wrong
        # service, so recreate rather than silently mismatch.
        print(f"♻️  '{name}' points at {current}; recreating it for {target}")
        zrok("release", name)

    rc, out = zrok("reserve", "public", target, "-n", name)
    if rc == 0:
        return "created"

    # 409: the name belongs to an environment this runtime cannot use, usually one that
    # was reset locally. The reservation is ours, so release it account-wide and retry.
    if "409" in out or "conflict" in out.lower():
        print(f"♻️  '{name}' is held by a previous environment; releasing it")
        zrok("release", name)
        rc, out = zrok("reserve", "public", target, "-n", name)
        if rc == 0:
            return "recreated"

    if looks_unauthorized(out):
        # Not a quota or naming problem: the environment itself is rejected, and an
        # ephemeral share would fail the same way. Let the caller re-enable and retry.
        raise PermissionError(out.strip())

    print("⚠️ Could not create the reserved share.")
    print(f"   {account_summary()}")
    print(f"   zrok said: {out.strip().splitlines()[-1] if out.strip() else '(no output)'}")
    if "500" in out:
        # zrok returns 500 rather than a clean quota error once the account is full, and
        # every Colab runtime that enables adds an environment.
        print("   A 500 here usually means the account is out of environment or share")
        print("   slots. Delete unused environments at https://api.zrok.io and re-run.")
    return None


def start_share(args, timeout=90):
    """Starts a share and returns (process, public_url).

    Output is drained on a worker thread so a zrok that starts but never speaks cannot
    hang the cell.
    """
    proc = subprocess.Popen(
        [ZROK_BIN, "share", *args, "--headless"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    lines = queue.Queue()

    def pump():
        for line in iter(proc.stdout.readline, ""):
            lines.put(line.rstrip())
        lines.put(None)

    threading.Thread(target=pump, daemon=True).start()

    seen = []
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            line = lines.get(timeout=1.0)
        except queue.Empty:
            if proc.poll() is not None:
                break
            continue
        if line is None:
            break

        seen.append(line)
        print(line)
        payload = zrok_json(line)
        text = payload.get("msg", "") if payload else line
        for word in text.split():
            if word.startswith("https://"):
                return proc, word.rstrip(".,'\"")

    proc.terminate()
    raise RuntimeError(
        "The share started but never reported a public URL.\n\nLast output:\n"
        + "\n".join(seen[-15:] or ["(no output)"]))


# Stop a tunnel this notebook started earlier, so re-running does not leave orphans.
previous = globals().get("unity_zrok_process")
if previous is not None and previous.poll() is None:
    print("♻️  stopping the previous tunnel from this notebook")
    previous.terminate()
    try:
        previous.wait(timeout=5)
    except subprocess.TimeoutExpired:
        previous.kill()

def open_tunnel():
    """One full attempt: reserve if possible, then share. Raises PermissionError on 401."""
    reservation = ensure_reserved(RESERVED_NAME, TARGET)

    if reservation is not None:
        print(f"🔖 reserved share '{RESERVED_NAME}': {reservation}")
        try:
            proc, url = start_share(["reserved", RESERVED_NAME])
            return proc, url, True
        except RuntimeError as exc:
            if looks_unauthorized(str(exc)):
                raise PermissionError(str(exc)) from exc
            print(f"⚠️ The reserved share would not start: {exc}")

    # An ephemeral share gets a random hostname but works the same. Unity's Colab Router
    # URL is just a text field, so a one-off URL costs a paste, not a broken session.
    print(f"🔒 Falling back to a one-off share for {TARGET} …")
    try:
        proc, url = start_share(["public", TARGET])
    except RuntimeError as exc:
        if looks_unauthorized(str(exc)):
            raise PermissionError(str(exc)) from exc
        raise
    return proc, url, False


print("🔑 zrok environment:", enable_environment(ZROK_ENABLE_TOKEN))

try:
    unity_zrok_process, public_url, stable_url = open_tunnel()
except PermissionError as exc:
    # zrok answers overview but rejects shares: the local environment is no longer
    # registered. Enabling a fresh one is the only repair from here.
    print(f"♻️  zrok rejected this environment ({str(exc).splitlines()[-1][:70]})")
    print("🔑 zrok environment:", enable_environment(ZROK_ENABLE_TOKEN, force=True))
    unity_zrok_process, public_url, stable_url = open_tunnel()

print(f"\n🚀 PUBLIC ENDPOINT: {public_url}")
print("   Paste this into the Spatial Generation window as the Colab Router URL.")
if not stable_url:
    print("   ⚠️ This URL is temporary and changes each run; the reserved name was unavailable.")


## Step 10 · Health check

Confirms the router and ComfyUI before you switch to Unity.


In [ ]:
# Confirm both services before switching to Unity.
import requests

def check(label, url):
    try:
        r = requests.get(url, timeout=10)
        return r.ok, f"{label}: {r.status_code} {r.text[:200]}"
    except requests.RequestException as exc:
        return False, f"{label}: unreachable ({exc})"

router_ok, router_msg = check("Router  ", "http://127.0.0.1:8000/health")
comfy_ok,  comfy_msg  = check("ComfyUI ", "http://127.0.0.1:8188/system_stats")
print(router_msg)
print(comfy_msg)

if router_ok:
    # The router reports ComfyUI too, and that is the view Unity acts on.
    health = requests.get("http://127.0.0.1:8000/health", timeout=10).json()
    if not health.get("comfy_reachable", True):
        print(f"\n⚠️ The router cannot reach ComfyUI at {health.get('comfy_url')}: {health.get('detail','')}")
        print("   Re-run the ComfyUI launch cell and wait for 'ComfyUI is running on port 8188'.")
    elif comfy_ok:
        print("\n✅ Both services are up. Point Unity at the public endpoint above.")
else:
    print("\n⚠️ Run the router cell, then the cell that serves it on port 8000.")


## Step 11 · Live ComfyUI log

Optional: watch a generation as it runs.


In [ ]:
# Live ComfyUI log tail, useful while a Unity generation is running.
#
# Stop it with the interrupt button; it also stops on its own at TAIL_SECONDS.
import os
import time

LOG_PATH = "/tmp/spatialgen_comfyui.log"
TAIL_SECONDS = 1800
DONE_MARKERS = ("Prompt executed in", "got prompt")

deadline = time.time() + TAIL_SECONDS
log_pos = max(0, os.path.getsize(LOG_PATH) - 4000) if os.path.exists(LOG_PATH) else 0

try:
    while time.time() < deadline:
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
                f.seek(log_pos)
                chunk = f.read()
                log_pos = f.tell()
            if chunk:
                print(chunk, end="")
        time.sleep(2)
except KeyboardInterrupt:
    print("\n⏹️ Log tail stopped.")
